# P5-3 · GĐ 3 · Điều khiển bằng chữ

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import math, torch, torch.nn as nn

def time_emb(t, dim=32):                              # như GĐ1
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half) / half)
    ang = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    return torch.cat([ang.sin(), ang.cos()], dim=1)

class CondDenoiser(nn.Module):
    def __init__(self, n_cls=3, h=256):
        super().__init__()
        self.lab = nn.Embedding(n_cls + 1, 32)          # chỉ số n_cls là nhãn rỗng ∅
        self.net = nn.Sequential(nn.Linear(2 + 32, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t, y):
        return self.net(torch.cat([x, time_emb(t) + self.lab(y)], 1))   # cộng nhãn vào thời gian

model = CondDenoiser()
x, t, y = torch.randn(5, 2), torch.randint(0, 1000, (5,)), torch.tensor([0, 1, 2, 0, 3])   # nhãn 3 = ∅
model(x, t, y).shape

torch.Size([5, 2])


In [ ]:
NULL_ID = 3
y = torch.randint(0, 3, (10_000,))                                         # nhãn thật của một batch lớn
y = torch.where(torch.rand(len(y)) < 0.1, torch.full_like(y, NULL_ID), y)   # 10% bỏ nhãn
(y == NULL_ID).float().mean().item()                                       # khoảng 0.1

0.10050000250339508


In [ ]:
x, t, w = torch.randn(4, 2), torch.full((4,), 500), 3.0
y, y_null = torch.zeros(4, dtype=torch.long), torch.full((4,), NULL_ID)
e_cond, e_uncond = model(x, t, y), model(x, t, y_null)
eps = e_uncond + w * (e_cond - e_uncond)
eps.shape                                  # cùng shape như một lần gọi model

torch.Size([4, 2])


## Bài tập 4: Diffusion có điều kiện và guidance trên điểm 2D {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
tam = torch.tensor([[-2.0, 0.0], [2.0, 0.0], [0.0, 2.5]])
y = torch.randint(0, 3, (3000,))
X = tam[y] + 0.6 * torch.randn(3000, 2)
T = 100
betas = torch.linspace(1e-4, 0.05, T); alphas = 1 - betas; abar = torch.cumprod(alphas, 0)
RONG = 3

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(4, 16)
        self.mlp = nn.Sequential(nn.Linear(2 + 1 + 16, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2))
    def forward(self, x, t, c):
        return self.mlp(torch.cat([x, t.unsqueeze(1) / T, self.emb(c)], 1))
net = Net()
# TODO: train 3000 bước có bỏ nhãn 15%; hàm sinh(lop, w); dung_cum[w], do_trai[w]

In [ ]:
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
for step in range(3000):
    i = torch.randint(0, 3000, (256,))
    x0, c = X[i], torch.where(torch.rand(256) < 0.15, RONG, y[i])
    t = torch.randint(0, T, (256,)); eps = torch.randn_like(x0); a = abar[t].unsqueeze(1)
    loss = ((net(a.sqrt() * x0 + (1 - a).sqrt() * eps, t.float(), c) - eps) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def sinh(lop, w, n=600):
    torch.manual_seed(1)
    x = torch.randn(n, 2)
    cl, cr = torch.full((n,), lop), torch.full((n,), RONG)
    for t in reversed(range(T)):
        tt = torch.full((n,), float(t))
        e_l, e_r = net(x, tt, cl), net(x, tt, cr)
        e = e_r + w * (e_l - e_r)
        x = (x - betas[t] / (1 - abar[t]).sqrt() * e) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)
    return x

dung_cum, do_trai = {}, {}
for w in (0, 1, 4):
    x = sinh(0, w)
    ok = torch.cdist(x, tam).argmin(1) == 0
    dung_cum[w], do_trai[w] = ok.float().mean().item(), x[ok].std(0).mean().item()
    print(f"w = {w}: đúng cụm {dung_cum[w]:.0%}, độ trải {do_trai[w]:.2f}")

w = 0: đúng cụm 28%, độ trải 0.70
w = 1: đúng cụm 100%, độ trải 0.63
w = 4: đúng cụm 100%, độ trải 1.12


## Bài tập 5: Loss tương phản của CLIP trên một batch {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn.functional as F
cap = ["con mèo trên ghế", "đĩa phở bò", "bãi biển lúc hoàng hôn", "con chó chạy trên cỏ"]
I = torch.tensor([[0.9, 0.1, 0.0], [0.0, 1.0, 0.1], [0.1, 0.2, 1.0], [0.7, 0.0, 0.5]])
Tx = torch.tensor([[1.0, 0.0, 0.1], [0.1, 0.9, 0.0], [0.0, 0.1, 0.9], [0.6, 0.1, 0.6]])
# TODO: chuẩn hóa, S, loss, so_dung

In [ ]:
In, Tn = F.normalize(I, dim=1), F.normalize(Tx, dim=1)
S = In @ Tn.T / 0.1
nhan = torch.arange(4)
loss = (F.cross_entropy(S, nhan) + F.cross_entropy(S.T, nhan)) / 2
so_dung = (S.argmax(1) == nhan).sum()
print((In @ Tn.T).round(decimals=2))
print(f"loss {loss.item():.3f}, ảnh tìm đúng chú thích: {so_dung.item()}/4")
for i, j in enumerate(S.argmax(1).tolist()):
    print(f"ảnh {i} → '{cap[j]}'")

tensor([[0.9900, 0.2200, 0.0100, 0.7100],
        [0.0100, 0.9900, 0.2100, 0.1900],
        [0.1900, 0.2000, 0.9900, 0.7800],
        [0.8700, 0.0900, 0.5800, 0.9800]])
loss 0.116, ảnh tìm đúng chú thích: 4/4
ảnh 0 → 'con mèo trên ghế'
ảnh 1 → 'đĩa phở bò'
ảnh 2 → 'bãi biển lúc hoàng hôn'
ảnh 3 → 'con chó chạy trên cỏ'


## Bài tập 6: Cross-attention: mỗi vùng ảnh hỏi chữ {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
feat = torch.randn(1, 64, 4, 4)
tokens = ["một", "con", "mèo", "đen", "trên", "ghế", "đỏ"]
text = torch.randn(1, 7, 32)
attn = nn.MultiheadAttention(embed_dim=64, num_heads=4, kdim=32, vdim=32, batch_first=True)
# TODO: q = feat duỗi thành (1, 16, 64); out, w = attn(q, text, text); out_img (1, 64, 4, 4)

In [ ]:
q = feat.flatten(2).transpose(1, 2)
out, w = attn(q, text, text)
out_img = out.transpose(1, 2).reshape(1, 64, 4, 4)
print("q:", tuple(q.shape), "| trọng số:", tuple(w.shape), "| ra:", tuple(out_img.shape))
print("vị trí (0, 0) chú ý nhiều nhất vào:", tokens[w[0, 0].argmax()])

q: (1, 16, 64) | trọng số: (1, 16, 7) | ra: (1, 64, 4, 4)
vị trí (0, 0) chú ý nhiều nhất vào: con
